# Project 3: Heart Disease Risk Assessment (Tree-based Models)

**Dataset:** UCI Heart Disease Dataset
**Goal:** Build Decision Tree and Random Forest classifiers to predict presence of heart disease, and identify the most important risk factors.

This notebook covers:
1. Data loading and cleaning
2. Exploratory Data Analysis (EDA)
3. Preprocessing (missing values, encoding)
4. Decision Tree model + visualization
5. Random Forest model
6. Comparing Single Tree vs Forest performance
7. Feature Importance analysis


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)


## 2. Load the Dataset

In [ ]:
df = pd.read_csv('heart_disease_uci.csv')
print("Shape of dataset:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
# Quick look at missing values in each column
df.isnull().sum().sort_values(ascending=False)


The dataset has quite a few missing values, especially in `ca`, `thal` and `slope`. It also
combines patients from 4 different hospitals (Cleveland, Hungary, Switzerland, VA Long Beach), which
is likely why so much data is missing - not every hospital recorded every test.

The `num` column is the original target and ranges from 0 (no disease) to 4 (different severity levels
of disease). For this project we only care about presence vs absence of heart disease, so it will be
converted into a binary target.

## 3. Data Cleaning & Target Creation

In [ ]:
# id column is just a row identifier, not useful for modeling
df = df.drop(columns=['id'])

# Convert target to binary: 0 = no heart disease, 1 = heart disease present (any severity)
df['target'] = (df['num'] > 0).astype(int)
df = df.drop(columns=['num'])

df['target'].value_counts()


In [ ]:
# Fill missing numeric columns with median (robust to outliers)
numeric_cols = ['trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill missing categorical columns with the most frequent value (mode)
categorical_cols = ['fbs', 'restecg', 'exang', 'slope', 'thal']
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Remaining missing values:", df.isnull().sum().sum())


In [ ]:
# fbs and exang are stored as TRUE/FALSE strings, convert to 0/1
df['fbs'] = df['fbs'].astype(str).map({'TRUE': 1, 'FALSE': 0})
df['exang'] = df['exang'].astype(str).map({'TRUE': 1, 'FALSE': 0})
df[['fbs', 'exang']].head()


## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='target', data=df, palette='Set2')
plt.title('Heart Disease Presence (0 = No, 1 = Yes)')
plt.xlabel('Target')
plt.ylabel('Number of Patients')
plt.show()


In [ ]:
plt.figure(figsize=(7,5))
sns.histplot(data=df, x='age', hue='target', kde=True, bins=25, palette='Set1')
plt.title('Age Distribution by Heart Disease Status')
plt.show()


In [ ]:
plt.figure(figsize=(7,5))
sns.boxplot(x='target', y='chol', data=df, palette='Set3')
plt.title('Cholesterol Levels by Heart Disease Status')
plt.show()


In [ ]:
plt.figure(figsize=(7,5))
sns.countplot(x='cp', hue='target', data=df, palette='Set2')
plt.title('Chest Pain Type vs Heart Disease')
plt.xticks(rotation=20)
plt.show()


In [ ]:
plt.figure(figsize=(9,7))
corr = df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap (numeric features)')
plt.show()


From the plots above:
- Heart disease becomes more common with age, especially past 50.
- Patients with heart disease tend to have a lower max heart rate achieved (`thalch`) and higher `oldpeak`.
- `cp` (chest pain type) looks like a strong indicator - asymptomatic patients show up much more often in the disease group, which matches medical intuition (silent/asymptomatic cases are often more serious).

## 5. Encoding Categorical Variables

In [ ]:
# One-hot encode remaining categorical (text) columns
df_encoded = pd.get_dummies(df, columns=['sex', 'dataset', 'cp', 'restecg', 'slope', 'thal'], drop_first=True)

X = df_encoded.drop(columns=['target'])
y = df_encoded['target']

print("Final feature count:", X.shape[1])
X.head()


## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## 7. Decision Tree Classifier

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

print("Decision Tree Accuracy :", accuracy_score(y_test, dt_pred))
print("Decision Tree Precision:", precision_score(y_test, dt_pred))
print("Decision Tree Recall   :", recall_score(y_test, dt_pred))
print("\nClassification Report:\n", classification_report(y_test, dt_pred))


### Visualizing the Decision Tree
Depth was limited to 4 so the tree stays readable and doesn't overfit.

In [ ]:
plt.figure(figsize=(20,10))
plot_tree(dt_model, feature_names=X.columns, class_names=['No Disease', 'Disease'],
          filled=True, rounded=True, fontsize=8)
plt.title('Decision Tree Visualization (max_depth=4)')
plt.show()


In [ ]:
cm_dt = confusion_matrix(y_test, dt_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Disease','Disease'], yticklabels=['No Disease','Disease'])
plt.title('Decision Tree - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


## 8. Random Forest Classifier

In [ ]:
rf_model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy :", accuracy_score(y_test, rf_pred))
print("Random Forest Precision:", precision_score(y_test, rf_pred))
print("Random Forest Recall   :", recall_score(y_test, rf_pred))
print("\nClassification Report:\n", classification_report(y_test, rf_pred))


In [ ]:
cm_rf = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Disease','Disease'], yticklabels=['No Disease','Disease'])
plt.title('Random Forest - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


## 9. Comparing Single Tree vs Random Forest

In [ ]:
results = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest'],
    'Accuracy': [accuracy_score(y_test, dt_pred), accuracy_score(y_test, rf_pred)],
    'Precision': [precision_score(y_test, dt_pred), precision_score(y_test, rf_pred)],
    'Recall': [recall_score(y_test, dt_pred), recall_score(y_test, rf_pred)],
    'F1-Score': [f1_score(y_test, dt_pred), f1_score(y_test, rf_pred)]
})
results


In [ ]:
results_melted = results.melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(9,5))
sns.barplot(x='Metric', y='Score', hue='Model', data=results_melted, palette='muted')
plt.title('Decision Tree vs Random Forest - Performance Comparison')
plt.ylim(0, 1)
plt.show()


The Random Forest outperforms the single Decision Tree across all metrics. This makes sense - a
single tree can overfit or make mistakes based on a few splits, while the forest averages predictions
from many trees (each trained on a random subset of data/features), which reduces variance and makes
the model more robust to noise in the data.

## 10. Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8,8))
sns.barplot(x=importances.values[:15], y=importances.index[:15], palette='viridis')
plt.title('Top 15 Feature Importances - Random Forest')
plt.xlabel('Importance Score')
plt.show()

importances.head(15)


**Key risk factors identified by the model:**

Based on the feature importance scores, `cp` (chest pain type), `thal` (thalassemia test result),
`oldpeak` (ST depression induced by exercise), `ca` (number of major vessels), age and max heart rate
(`thalch`) come out as the strongest predictors of heart disease in this dataset - which lines up well
with what's known clinically about cardiac risk factors.

## 11. Conclusion

- The dataset required a fair amount of cleaning (missing values in `ca`, `thal`, `slope` etc., and
  converting the multi-class `num` target into a binary target).
- The **Random Forest model performed better than a single Decision Tree** on accuracy, precision,
  recall and F1-score, confirming that ensembling reduces overfitting and improves robustness.
- Chest pain type, thal, oldpeak, number of major vessels (ca), and age turned out to be the most
  important features for predicting heart disease presence.
- Possible future improvements: hyperparameter tuning (GridSearchCV), trying Gradient Boosting/XGBoost,
  and handling the class imbalance more carefully.